Facial Feature Faster R-CNN Training Pipeline

fasterrcnn_resnet50_fpn_v2 is an advanced object detection model that combines the Faster R-CNN framework with a ResNet-50 backbone and a Feature Pyramid Network (FPN) for feature extraction. It is designed to provide high accuracy in detecting objects in images, especially in scenarios with small or overlapping objects.

In [ ]:
import os
import yaml
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import time
from collections import defaultdict
import numpy as np
import cv2
import matplotlib.pyplot as plt
from copy import deepcopy
# Use the new GradScaler import if available (PyTorch 2.0+)
try:
    from torch.amp import GradScaler
except ImportError:
    from torch.cuda.amp import GradScaler

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
itrain = r'D:\Hassan Ali\Projects\Machine Learning\Features of Face Extraction\Facial Feature Extraction Dataset\train'
ivalid = r'D:\Hassan Ali\Projects\Machine Learning\Features of Face Extraction\Facial Feature Extraction Dataset\valid'
itest  = r'D:\Hassan Ali\Projects\Machine Learning\Features of Face Extraction\Facial Feature Extraction Dataset\test'

In [ ]:

face_yaml = dict(
    train = itrain,
    val   = ivalid,
    test  = itest,
    nc    = 5,
    names = ['eyes', 'eyebrows', 'lips', 'mustache-beard', 'nose']
)

with open('face.yaml', 'w') as outfile:
    yaml.dump(face_yaml, outfile, default_flow_style=True)

In [ ]:
class YoloToFasterRCNNDataset(Dataset):
    """Dataset class that converts YOLO annotations to Faster R-CNN targets with error handling and augmentation."""
    def __init__(self, images_dir, labels_dir, class_names, transforms=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.class_names = class_names
        self.image_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
        self.label_files = [f.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt')
                            for f in self.image_files]
        
        # Validate existence of label files
        for lbl_file in self.label_files:
            label_path = os.path.join(labels_dir, lbl_file)
            if not os.path.exists(label_path):
                raise FileNotFoundError(f"Label file {lbl_file} not found in {labels_dir}!")
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        try:
            # Load image
            img_path = os.path.join(self.images_dir, self.image_files[idx])
            img = Image.open(img_path).convert("RGB")
            img_width, img_height = img.size
        except Exception as e:
            print(f"Error reading image {img_path}: {e}")
            raise e
        
        # Load labels
        boxes, labels = [], []
        label_path = os.path.join(self.labels_dir, self.label_files[idx])
        try:
            with open(label_path, "r") as f:
                for line in f.readlines():
                    if line.strip():
                        parts = line.strip().split()
                        if len(parts) != 5:
                            continue  # Skip invalid lines
                        class_id, x_center, y_center, width, height = map(float, parts)
                        # Convert YOLO to Pascal VOC coordinates
                        xmin = max(0, (x_center - width / 2) * img_width)
                        ymin = max(0, (y_center - height / 2) * img_height)
                        xmax = min(img_width, (x_center + width / 2) * img_width)
                        ymax = min(img_height, (y_center + height / 2) * img_height)
                        
                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(int(class_id) + 1)  # +1 so that class indices start at 1
        except Exception as e:
            print(f"Error reading label file {label_path}: {e}")
            raise e

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

def load_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

def get_model(num_classes):
    """Load a pre-trained Faster R-CNN model and replace the head for our number of classes."""
    # Use the new weights API instead of 'pretrained'
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
    model = fasterrcnn_resnet50_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def get_transform(train=True):
    """Compose data transforms. Augmentations (for training) are applied before converting to a tensor."""
    transform_list = []
    if train:
        transform_list.extend([
            torchvision.transforms.RandomHorizontalFlip(0.5),
            torchvision.transforms.RandomRotation(10),
            torchvision.transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        ])
    transform_list.append(torchvision.transforms.ToTensor())
    return torchvision.transforms.Compose(transform_list)

class AverageMeter:
    """Compute and store the average and current value of a metric."""
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch, scaler=None, print_freq=10):
    """Train the model for one epoch using mixed-precision."""
    model.train()
    loss_meter = AverageMeter()
    start_time = time.time()
    
    for batch_idx, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loss_meter.update(losses.item())
        
        if batch_idx % print_freq == 0:
            elapsed = time.time() - start_time
            print(f'Epoch: [{epoch}][{batch_idx}/{len(data_loader)}]  '
                  f'Loss: {loss_meter.avg:.4f}  '
                  f'Time: {elapsed:.2f}s')
    
    return loss_meter.avg

def compute_iou(box1, box2):
    # box format: [xmin, ymin, xmax, ymax]
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    denominator = box1_area + box2_area - intersection
    return intersection / denominator if denominator > 0 else 0

def evaluate(model, data_loader, device, iou_threshold=0.5):
    """Evaluate the model on a validation set and compute per-class precision, recall, f1 and overall mAP."""
    model.eval()
    
    class_metrics = defaultdict(lambda: {'true_positives': 0, 'false_positives': 0, 'false_negatives': 0})
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            outputs = model(images)
            
            for target, output in zip(targets, outputs):
                # Process each image
                pred_boxes = output['boxes'].cpu()
                pred_labels = output['labels'].cpu()
                pred_scores = output['scores'].cpu()
                
                true_boxes = target['boxes'].cpu()
                true_labels = target['labels'].cpu()
                
                # For each class present in the ground truth
                for label in torch.unique(true_labels):
                    class_key = str(label.item())
                    
                    true_class_mask = true_labels == label
                    true_class_boxes = true_boxes[true_class_mask]
                    
                    pred_class_mask = pred_labels == label
                    pred_class_boxes = pred_boxes[pred_class_mask]
                    pred_class_scores = pred_scores[pred_class_mask]
                    
                    if len(pred_class_boxes) == 0:
                        class_metrics[class_key]['false_negatives'] += len(true_class_boxes)
                        continue
                    
                    # Sort predictions by score
                    sorted_idx = torch.argsort(pred_class_scores, descending=True)
                    pred_class_boxes = pred_class_boxes[sorted_idx]
                    
                    used_predictions = set()
                    
                    # Match each ground truth box to the best prediction
                    for gt_idx, gt_box in enumerate(true_class_boxes):
                        best_iou = iou_threshold
                        best_pred_idx = None
                        
                        for pred_idx, pred_box in enumerate(pred_class_boxes):
                            if pred_idx in used_predictions:
                                continue
                            iou = compute_iou(gt_box.numpy(), pred_box.numpy())
                            if iou > best_iou:
                                best_iou = iou
                                best_pred_idx = pred_idx
                        if best_pred_idx is not None:
                            class_metrics[class_key]['true_positives'] += 1
                            used_predictions.add(best_pred_idx)
                        else:
                            class_metrics[class_key]['false_negatives'] += 1
                    
                    # Any remaining predictions are false positives
                    class_metrics[class_key]['false_positives'] += len(pred_class_boxes) - len(used_predictions)
    
    # Compute per-class precision, recall, f1
    results = {}
    precisions = []
    for class_key, metrics in class_metrics.items():
        tp = metrics['true_positives']
        fp = metrics['false_positives']
        fn = metrics['false_negatives']
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        results[class_key] = {
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
        precisions.append(precision)
    
    # Compute mean average precision (mAP) as the average of per-class precisions.
    results['map'] = sum(precisions) / len(precisions) if precisions else 0
    return results

In [ ]:
def train_model(config_path, num_epochs=10, batch_size=2):
    config = load_config(config_path)
    train_path = config['train']
    val_path   = config['val']
    num_classes = config['nc'] + 1  # +1 for the background class
    class_names = config['names']
    
    # Create datasets
    train_dataset = YoloToFasterRCNNDataset(
        os.path.join(train_path, 'images'),
        os.path.join(train_path, 'labels'),
        class_names,
        transforms=get_transform(train=True)
    )
    val_dataset = YoloToFasterRCNNDataset(
        os.path.join(val_path, 'images'),
        os.path.join(val_path, 'labels'),
        class_names,
        transforms=get_transform(train=False)
    )
    
    # Set num_workers=0 on Windows to avoid multiprocessing issues
    num_workers = 0 if os.name == 'nt' else 4
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=num_workers)
    val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False,
                              collate_fn=collate_fn, num_workers=num_workers)
    
    # Prepare model, optimizer, and mixed-precision scaler (enabled only if CUDA is available)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_model(num_classes).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    # Early stopping variables
    best_mAP = 0.0
    best_weights = None
    patience = 3
    epochs_no_improve = 0
    
    for epoch in range(num_epochs):
        # Train one epoch
        train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch, scaler)
        
        # Validate the model
        metrics = evaluate(model, val_loader, device)
        current_mAP = metrics.get('map', 0)
        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, mAP = {current_mAP:.4f}")
        
        # Early stopping check
        if current_mAP > best_mAP:
            best_mAP = current_mAP
            best_weights = deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break
        
        # Save checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': metrics,
        }, f'checkpoint_epoch_{epoch}.pth')
    
    # Load the best weights and save the final model
    if best_weights is not None:
        model.load_state_dict(best_weights)
    torch.save(model.state_dict(), 'best_model.pth')
    return model

In [ ]:
def save_as_h5(pytorch_model, input_shape=(3, 224, 224)):
    """Export the PyTorch model to H5 format via ONNX and TensorFlow conversion."""
    device_local = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dummy_input = torch.randn(1, *input_shape).to(device_local)
    onnx_path = "model.onnx"
    
    # Export to ONNX
    torch.onnx.export(
        pytorch_model,
        dummy_input,
        onnx_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    )
    
    # Convert the ONNX model to TensorFlow
    import onnx
    from onnx_tf.backend import prepare
    onnx_model = onnx.load(onnx_path)
    tf_rep = prepare(onnx_model)
    tf_rep.export_graph("tf_model")
    
    import tensorflow as tf
    model_tf = tf.saved_model.load("tf_model")
    # Attempt to save as H5 only if model_tf is a Keras model
    if hasattr(model_tf, 'save'):
        model_tf.save("model.h5")
    else:
        print("Converted TensorFlow model is not a Keras model and cannot be saved as .h5 format directly.")

In [ ]:
def inference(model, image_path, class_names, device, confidence_threshold=0.5):
    """Run inference on a single image and return bounding boxes, class names, and scores."""
    model.eval()
    img = Image.open(image_path).convert("RGB")
    # Use non-training transforms for inference
    transform = get_transform(train=False)
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        prediction = model(img_tensor)
        
    boxes = prediction[0]['boxes'].cpu().numpy()
    labels = prediction[0]['labels'].cpu().numpy()
    scores = prediction[0]['scores'].cpu().numpy()
    
    mask = scores >= confidence_threshold
    boxes = boxes[mask]
    labels = labels[mask]
    scores = scores[mask]
    
    label_names = [class_names[label - 1] for label in labels]
    
    return boxes, label_names, scores

def get_color_map(class_names):
    """Assign a fixed random color to each class (for reproducible visualization)."""
    np.random.seed(42)
    colors = {}
    for i, name in enumerate(class_names):
        colors[name] = tuple(map(int, np.random.randint(0, 255, 3)))
    return colors

def visualize_predictions(image_path, boxes, labels, scores, class_names, confidence_threshold=0.5):
    """Draw bounding boxes and labels on the image."""
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
    colors = get_color_map(class_names)
    
    for box, label, score in zip(boxes, labels, scores):
        if score < confidence_threshold:
            continue
        x1, y1, x2, y2 = map(int, box)
        color = colors[label]
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        text = f'{label}: {score:.2f}'
        (text_width, text_height), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(image, (x1, y1 - text_height - baseline - 5), (x1 + text_width, y1), color, -1)
        cv2.putText(image, text, (x1, y1 - baseline - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    return image

def save_visualization(image, output_path):
    """Save the image with visualized predictions."""
    cv2.imwrite(output_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))

def show_visualization(image):
    """Display the image with visualized predictions."""
    plt.figure(figsize=(12, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

In [ ]:

def inference_with_visualization(model, image_path, class_names, device, confidence_threshold=0.5, 
                                 visualize=True, save_path=None):
    """Perform inference and optionally visualize the results."""
    model.eval()
    img = Image.open(image_path).convert("RGB")
    transform = get_transform(train=False)
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        prediction = model(img_tensor)
        
    boxes = prediction[0]['boxes'].cpu().numpy()
    labels = prediction[0]['labels'].cpu().numpy()
    scores = prediction[0]['scores'].cpu().numpy()
    
    mask = scores >= confidence_threshold
    boxes = boxes[mask]
    labels = labels[mask]
    scores = scores[mask]
    
    label_names = [class_names[label - 1] for label in labels]
    
    vis_image = None
    if visualize:
        vis_image = visualize_predictions(image_path, boxes, label_names, scores, class_names, confidence_threshold)
        show_visualization(vis_image)
        if save_path:
            save_visualization(vis_image, save_path)
    
    return boxes, label_names, scores, vis_image


In [ ]:
def main():
    config_path = 'face.yaml'
    model_path = 'best_model.pth'  # Path to your saved checkpoint
    
    config = load_config(config_path)
    class_names = config['names']
    num_classes = config['nc'] + 1  # +1 for background class
    
    # Initialize model architecture
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_model(num_classes).to(device)
    
    # Load pre-trained weights if they exist
    if os.path.exists(model_path):
        print(f"Loading pre-trained weights from {model_path}")
        model.load_state_dict(torch.load(model_path, map_location=device))
    else:
        print("No pre-trained weights found. Training from scratch...")
        num_epochs = 10
        batch_size = 2
        model = train_model(config_path, num_epochs, batch_size)
    
    # Perform inference on test set
    if 'test' in config:
        print("Running inference with visualization on test set...")
        test_dataset = YoloToFasterRCNNDataset(
            images_dir=os.path.join(config['test'], 'images'),
            labels_dir=os.path.join(config['test'], 'labels'),
            class_names=class_names,
            transforms=get_transform(train=False)
        )
        
        if len(test_dataset) > 0:
            for i in range(min(5, len(test_dataset))):  # Process first 5 test images
                test_image_path = os.path.join(config['test'], 'images', test_dataset.image_files[i])
                save_path = f'prediction_result_{i}.jpg'
                
                boxes, labels, scores, vis_image = inference_with_visualization(
                    model, 
                    test_image_path, 
                    class_names, 
                    device,
                    confidence_threshold=0.5,
                    visualize=True,
                    save_path=save_path
                )
                
                print(f"\nResults for test image {i + 1}:")
                for box, label, score in zip(boxes, labels, scores):
                    print(f"Class: {label}, Score: {score:.3f}")
                    print(f"Box coordinates [xmin, ymin, xmax, ymax]: {box}")
    
    return model
if __name__ == "__main__":
    trained_model = main()
    if trained_model is not None:
        save_as_h5(trained_model)  # Optional: Export to H5
    else:
        print("Model not loaded/exported.")